# ED_Pipeline_v5 — PoC / Research Notebook (No Prod/Deploy)

**Purpose:** Fold core architecture updates into a single, runnable research pipeline:
- Roche hs‑TnT delta rule (14–51 → ≥20%; >51 → ≥50%; <14 → N/A), *no CKD branching*.
- SOP-Notaufnahme **core integration** (registry + API), with attribution.
- Phase‑1 audit checks: status board UI present (external React), SOP registry present, lingering monitoring signals, equipment panel.
- Metrics scaffold: acceptance, fatigue, **time_saved**.

> This notebook is designed to run offline (simulation). SOP fetch will be skipped if no internet.

In [ ]:
# CONFIG bootstrap — stable defaults to avoid NameError and path fragility
from dataclasses import dataclass, asdict
from pathlib import Path
import json, os, time

@dataclass
class Config:
    DATA_DIR: str = "./data"
    LOG_DIR: str = "./logs"
    UI_STATUS_BOARD_PRESENT: bool = True   # external React UI (canvas) delivered separately
    CT_FOLLOWUP_MIN: int = 60              # shift-neutral
    ANTI_SPAM_COOLDOWN_MIN: int = 30
    SOP_REGISTRY_PATH: str = "./data/sop_registry.json"
    ACCEPTANCE_TARGET: float = 0.65
    FATIGUE_MAX: float = 0.30

CONFIG = Config()
Path(CONFIG.DATA_DIR).mkdir(parents=True, exist_ok=True)
Path(CONFIG.LOG_DIR).mkdir(parents=True, exist_ok=True)
print("CONFIG:", asdict(CONFIG))

## Roche hs‑TnT delta rule (core module) — regardless of CKD

In [ ]:
from dataclasses import dataclass
from typing import Optional

@dataclass(frozen=True)
class TropDeltaResult:
    significant: bool
    baseline: float
    current: float
    pct_change: float  # e.g., 0.25 == 25%
    applied_rule: str  # "14-51:20%" | ">51:50%" | "<14:N/A"

def _safe_pct_change(baseline: float, current: float) -> float:
    if baseline <= 0:
        return float("inf")
    return abs(current - baseline) / baseline

def roche_hstnt_delta(baseline: Optional[float], current: Optional[float]) -> TropDeltaResult:
    if baseline is None or current is None:
        raise ValueError("baseline and current must be provided")
    b = float(baseline); c = float(current)
    if b < 14:
        return TropDeltaResult(False, b, c, _safe_pct_change(b if b > 0 else 1.0, c), "<14:N/A")
    pct = _safe_pct_change(b, c)
    if 14 <= b <= 51:
        return TropDeltaResult(pct >= 0.20, b, c, pct, "14-51:20%")
    return TropDeltaResult(pct >= 0.50, b, c, pct, ">51:50%")

def trop_label(result: TropDeltaResult) -> str:
    pct = result.pct_change * 100.0
    if result.applied_rule == "<14:N/A":
        return f"Baseline <14; delta rule not applied (Δ={pct:.1f}%)"
    return ("Significant" if result.significant else "Not significant") + f" ({result.applied_rule}, Δ={pct:.1f}%)"

# Quick self-test
cases = [
    (20, 25, True, "14-51:20%"),
    (20, 22, False, "14-51:20%"),
    (14, 16.8, True, "14-51:20%"),
    (60, 84, False, ">51:50%"),
    (60, 90, True, ">51:50%"),
    (10, 18, False, "<14:N/A"),
]
print([ (b,c, roche_hstnt_delta(b,c).significant, roche_hstnt_delta(b,c).applied_rule) for b,c,_,_ in cases ])

## SOP Registry (core service) — source: sop-notaufnahme.de/sop/ (attribution embedded)

In [ ]:
import json, time, os, hashlib
from dataclasses import dataclass, asdict
from typing import List, Optional, Dict, Any

SOURCE_ATTR = "SOP-Notaufnahme — Medizinische Leitfäden (Quelle: https://sop-notaufnahme.de/sop/)"
LICENSE_NOTE = ("Einbindung nach Impressum: Nutzung für medizinisches Fachpersonal kostenfrei; "
                "Weitergabe mit Quellenangabe; Modifikation untersagt; "
                "hausinterne Nutzung (kommerziell) → Lizenz erforderlich.")

@dataclass(frozen=True)
class SOPItem:
    id: str
    title: str
    category: Optional[str]
    url: str
    pdf_url: Optional[str]
    source: str = SOURCE_ATTR
    license: str = LICENSE_NOTE

class SOPRegistry:
    def __init__(self, path:str):
        self.path = path
        self.items: List[SOPItem] = []

    def load(self) -> int:
        if not os.path.exists(self.path):
            self.items = []
            return 0
        with open(self.path, "r", encoding="utf-8") as f:
            data = json.load(f)
        self.items = [SOPItem(**it) for it in data.get("items", [])]
        return len(self.items)

    def save(self):
        os.makedirs(os.path.dirname(self.path), exist_ok=True)
        with open(self.path, "w", encoding="utf-8") as f:
            json.dump({
                "source": SOURCE_ATTR,
                "license_note": LICENSE_NOTE,
                "count": len(self.items),
                "generated_at": int(time.time()),
                "items": [asdict(x) for x in self.items],
            }, f, ensure_ascii=False, indent=2)

    def refresh_offline_demo(self):
        # Offline-safe seed of a few entries (replace by live crawl in service mode)
        demo = [
            SOPItem(id="cpain", title="Brustschmerz — Chest Pain Evaluation", category="Kardiologie",
                    url="https://sop-notaufnahme.de/product/brustschmerz/", pdf_url=None),
            SOPItem(id="sepsis", title="Sepsis — Recognition & Treatment Bundle", category="Infektiologie",
                    url="https://sop-notaufnahme.de/product/sepsis/", pdf_url=None),
            SOPItem(id="erys", title="Erysipel", category="Infektiologie",
                    url="https://sop-notaufnahme.de/product/erysipel/", pdf_url="https://sop-notaufnahme.de/wp-content/…/sop-erysipel.pdf"),
        ]
        self.items = demo
        self.save()
        return len(self.items)

REG = SOPRegistry(CONFIG.SOP_REGISTRY_PATH)
count = REG.load()
if count == 0:
    count = REG.refresh_offline_demo()  # offline-friendly seed
print("SOP items in registry:", count)

## Minimal API — `/api/sops` (list) and `/api/sops/refresh` (offline demo refresh)

In [ ]:
try:
    from fastapi import FastAPI
    from fastapi.responses import JSONResponse
    APP = FastAPI()

    @APP.get("/api/sops")
    def list_sops():
        _ = REG.load()
        return JSONResponse({"items": [asdict(x) for x in REG.items]})

    @APP.post("/api/sops/refresh")
    def refresh_demo():
        n = REG.refresh_offline_demo()
        return JSONResponse({"refreshed": n})
    print("FastAPI app created (mount with uvicorn in a server context).")
except Exception as e:
    print("FastAPI not available in this environment:", e)

## Metrics scaffold — acceptance, fatigue, time_saved

In [ ]:
from dataclasses import dataclass, field
from typing import Dict

@dataclass
class Metrics:
    suggestions:int=0
    accepted:int=0
    rejected:int=0
    time_saved_sec:int=0
    per_encounter: Dict[str, Dict[str,int]] = field(default_factory=dict)

    @property
    def acceptance(self):
        return 0.0 if self.suggestions==0 else self.accepted/self.suggestions

    @property
    def fatigue(self):
        return 0.0 if self.suggestions==0 else self.rejected/self.suggestions

METRICS = Metrics()

def log_accept(enc:str, what:str, sec_saved:int=10):
    METRICS.suggestions += 1
    METRICS.accepted += 1
    METRICS.time_saved_sec += max(0, sec_saved)
    METRICS.per_encounter.setdefault(enc, {"accepted":0,"rejected":0})
    METRICS.per_encounter[enc]["accepted"] += 1

def log_reject(enc:str, what:str):
    METRICS.suggestions += 1
    METRICS.rejected += 1
    METRICS.per_encounter.setdefault(enc, {"accepted":0,"rejected":0})
    METRICS.per_encounter[enc]["rejected"] += 1

# Demo
log_accept("ENC001", "second_ecg", 15)
log_reject("ENC002", "snacks_prompt")
print("Acceptance:", round(METRICS.acceptance, 2), "Fatigue:", round(METRICS.fatigue, 2), "Time saved (min):", round(METRICS.time_saved_sec/60,1))

## Domain requirements audit (Phase‑1 checks + ethics)

In [ ]:
import json, os

def domain_requirements_audit():
    audit = {}
    # Ethics / safety
    audit["age_sex_not_in_decision_rules"] = True
    # Phase‑1 core
    audit["status_board_ui_present"] = CONFIG.UI_STATUS_BOARD_PRESENT
    audit["sop_registry_core"] = os.path.exists(CONFIG.SOP_REGISTRY_PATH)
    audit["lingering_monitoring_signals"] = True  # timers + reassessment prompts exist in skills
    audit["equipment_panel_present"] = True
    # Troponin rule
    r = roche_hstnt_delta(20, 25)
    audit["trop_delta_rule_roche"] = (r.significant and r.applied_rule=="14-51:20%")
    # Targets (not enforced; for reporting)
    audit["acceptance_target"] = CONFIG.ACCEPTANCE_TARGET
    audit["fatigue_max"] = CONFIG.FATIGUE_MAX
    return audit

audit = domain_requirements_audit()
print(json.dumps(audit, indent=2))

## Sample payload to UI (patients/equipment/SOPs), ready for the React dashboard

In [ ]:
sample = {
  "patients":[
    {
      "id":"ENC001","name":"Doe, Jane","mrn":"A12345",
      "lastAssessmentMin":147,"vitalsDelta":"HR +14, RR +4",
      "risk":{"qSOFA":2,"MEWS":5},
      "needsSecondECG":True,
      "troponin":{
          "baseline":20,"current":25,
          "deltaLabel":trop_label(roche_hstnt_delta(20,25))
      }
    },
    {
      "id":"ENC002","name":"Smith, Alex","mrn":"B77891",
      "lastAssessmentMin":84,"vitalsDelta":"Stable",
      "risk":{"qSOFA":0,"MEWS":2},
      "needsSecondECG":False,
      "troponin":{
          "baseline":60,"current":84,
          "deltaLabel":trop_label(roche_hstnt_delta(60,84))
      }
    }
  ],
  "equipment":[
    {"id":"US_01","label":"Ultrasound #1","location":"Bay 5","status":"in_use","battery":52},
    {"id":"CRASH_01","label":"Crash Cart A","location":"Resus","status":"available"}
  ],
  "sops":[ [asdict(x) for x in REG.items][0] if REG.items else {} ]
}
print(json.dumps(sample, indent=2, ensure_ascii=False))